In [1]:
import kagglehub
from pathlib import Path

# Download latest version
path = kagglehub.dataset_download(
    handle="budincsevity/szeged-weather",
    output_dir=Path.cwd() / "data",
    force_download=True
)

print("Path to dataset files:", path)

d:\Programming\Deep Learning\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 2.23M/2.23M [00:01<00:00, 1.31MB/s]

Extracting files...
Path to dataset files: d:\Programming\Deep Learning\ANN & Backpropogation\data


In [9]:
import pandas as pd

df = pd.read_csv(
    "data/weatherHistory.csv",
    usecols=["Summary", "Precip Type", "Temperature (C)", "Apparent Temperature (C)", "Humidity", "Wind Speed (km/h)", "Wind Bearing (degrees)", "Visibility (km)", "Pressure (millibars)"]).dropna()
df.head()

,Summary,Precip Type,Temperature (C),Apparent Temperature (C),Humidity,Wind Speed (km/h),Wind Bearing (degrees),Visibility (km),Pressure (millibars)
0,Partly Cloudy,rain,9.472222,7.388889,0.89,14.1197,251.0,15.8263,1015.13
1,Partly Cloudy,rain,9.355556,7.227778,0.86,14.2646,259.0,15.8263,1015.63
2,Mostly Cloudy,rain,9.377778,9.377778,0.89,3.9284,204.0,14.9569,1015.94
3,Partly Cloudy,rain,8.288889,5.944444,0.83,14.1036,269.0,15.8263,1016.41
4,Mostly Cloudy,rain,8.755556,6.977778,0.83,11.0446,259.0,15.8263,1016.51


In [10]:
# Training and testing
from sklearn.model_selection import train_test_split

root_dir = Path.cwd()
train_df, test_df = train_test_split(df, test_size=0.25, random_state = 42)

train_df.to_csv(root_dir / "data" / "train.csv", index=False)
test_df.to_csv(root_dir / "data" / "test.csv", index=False)

In [11]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
encoder = OneHotEncoder(
    categories = [train_df["Summary"].unique().tolist(), train_df["Precip Type"].unique().tolist()], 
    sparse_output = False,
    handle_unknown = "ignore" # Found unknown categories ['Dangerously Windy and Partly Cloudy'] in column 0 during transform
)
encoder.fit(train_df.loc[:, ["Summary", "Precip Type"]])

,"categories categories: 'auto' or a list of array-like, default='auto'Categories (unique values) per feature:- 'auto' : Determine categories automatically from the training data.- list : ``categories[i]`` holds the categories expected in the ith column. The passed categories should not mix strings and numeric values within a single feature, and should be sorted in case of numeric values.The used categories can be found in the ``categories_`` attribute... versionadded:: 0.20","[['Overcast', 'Foggy', ...], ['rain', 'snow']]"
,"drop drop: {'first', 'if_binary'} or an array-like of shape (n_features,), default=NoneSpecifies a methodology to use to drop one of the categories perfeature. This is useful in situations where perfectly collinearfeatures cause problems, such as when feeding the resulting datainto an unregularized linear regression model.However, dropping one category breaks the symmetry of the originalrepresentation and can therefore induce a bias in downstream models,for instance for penalized linear classification or regression models.- None : retain all features (the default).- 'first' : drop the first category in each feature. If only one category is present, the feature will be dropped entirely.- 'if_binary' : drop the first category in each feature with two categories. Features with 1 or more than 2 categories are left intact.- array : ``drop[i]`` is the category in feature ``X[:, i]`` that should be dropped.When `max_categories` or `min_frequency` is configured to groupinfrequent categories, the dropping behavior is handled after thegrouping... versionadded:: 0.21 The parameter `drop` was added in 0.21... versionchanged:: 0.23 The option `drop='if_binary'` was added in 0.23... versionchanged:: 1.1 Support for dropping infrequent categories.",None
,"sparse_output sparse_output: bool, default=TrueWhen ``True``, it returns a :class:`scipy.sparse.csr_matrix`,i.e. a sparse matrix in ""Compressed Sparse Row"" (CSR) format... versionadded:: 1.2 `sparse` was renamed to `sparse_output`",False
,"dtype dtype: number type, default=np.float64Desired dtype of output.",<class 'numpy.float64'>
,"handle_unknown handle_unknown: {'error', 'ignore', 'infrequent_if_exist', 'warn'}, default='error'Specifies the way unknown categories are handled during :meth:`transform`.- 'error' : Raise an error if an unknown category is present during transform.- 'ignore' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will be all zeros. In the inverse transform, an unknown category will be denoted as None.- 'infrequent_if_exist' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will map to the infrequent category if it exists. The infrequent category will be mapped to the last position in the encoding. During inverse transform, an unknown category will be mapped to the category denoted `'infrequent'` if it exists. If the `'infrequent'` category does not exist, then :meth:`transform` and :meth:`inverse_transform` will handle an unknown category as with `handle_unknown='ignore'`. Infrequent categories exist based on `min_frequency` and `max_categories`. Read more in the :ref:`User Guide `.- 'warn' : When an unknown category is encountered during transform a warning is issued, and the encoding then proceeds as described for `handle_unknown=""infrequent_if_exist""`... versionchanged:: 1.1 `'infrequent_if_exist'` was added to automatically handle unknown categories and infrequent categories... versionadded:: 1.6 The option `""warn""` was added in 1.6.",'ignore'
,"min_frequency min_frequency: int or float, default=NoneSpecifies the minimum frequency below which a category will beconsidered infrequent.- If `int`, categories with a smaller cardinality will be considered infrequent.- If `float`, categories with a smaller cardinality than `min_frequency * n_samples` will be considered infrequent... versionadded:: 1.1 Read more i

In [12]:
scaler = StandardScaler()
scaler.fit(train_df.drop(labels=["Summary", "Precip Type", "Visibility (km)"], axis=1))

# Even though you split the data later, your scaler's mean and std are now calculated using values from the test set. This is a subtle form of data leakage. The Fix: You should split your data first, then call .fit() only on the training set.

,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


In [13]:
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

class CustomDataset(Dataset):
    def __init__(self, path):
        with open(path, mode='r') as f:
            self.df = pd.read_csv(f, usecols=["Summary", "Precip Type", "Temperature (C)", "Apparent Temperature (C)", "Humidity", "Wind Speed (km/h)", "Wind Bearing (degrees)", "Visibility (km)", "Pressure (millibars)"]).dropna()

            # Seperating features and labels
            self.features = self.df.drop(labels=["Visibility (km)"], axis=1)
            self.labels = self.df.loc[:, "Visibility (km)"].to_numpy() 
        
        # Encoding categorical columns
        self.encoded_categories = encoder.transform(self.features.loc[:, ["Summary", "Precip Type"]]) 
        self.features = self.features.drop(labels=["Summary", "Precip Type"], axis=1) # dropping categorical features

        # Standardizing the rest of the features
        self.features = scaler.transform(self.features) # returns numpy array

        # Dataset restructuring
        
        self.features = np.append(
            self.features,
            self.encoded_categories,
            axis = 1
        ) # You can use np.hstack() as well

        # Converting to tensors
        self.features = torch.tensor(self.features, dtype=torch.float)
        self.labels = torch.tensor(self.labels, dtype=torch.float)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        if index < self.labels.shape[0]:
            return self.features[index], self.labels[index]

In [14]:
train_df = CustomDataset(path="data/train.csv")
test_df = CustomDataset(path="data/test.csv")

In [21]:
train_dataloader = DataLoader(
    dataset=train_df,
    batch_size=100,
    shuffle=True
)
test_dataloader = DataLoader(
    dataset=test_df,
    batch_size=100,
    shuffle=False # In your test_dataloader, you have shuffle=True. Shuffling the test set isn't wrong, but it's unnecessary and usually set to False to make results more easily reproducible during debugging.
)

In [17]:
from torch.nn import GELU, Module, Sequential, Linear, LeakyReLU

class CustomModel(Module):
    def __init__(self, num_features) -> None:
        super().__init__()

        self.model = Sequential(
            Linear(in_features=num_features, out_features=10),
            GELU(),

            Linear(in_features=10, out_features=5),
            GELU(),

            Linear(in_features=5, out_features=1),
        )
    
    def forward(self, X):
        return self.model(X)

In [18]:
# defining parameters
learning_rate = 0.001
epochs = 10

In [19]:
model = CustomModel(num_features = test_df.features.shape[1])
model = model.to('cuda') # Moving the model to the GPU is an expensive operation. Doing it inside the inner loop (for every single batch) slows down training significantly. The Fix: Move the model to 'cuda' once, before the for i in range(10) loop starts.

criterion = torch.nn.modules.loss.HuberLoss(delta=1.0)
optimizer = torch.optim.Adam(params=model.parameters(), lr=learning_rate)

In [22]:
# Training
for i in range(25):
    total_loss = 0
    for features, labels in train_dataloader:
        optimizer.zero_grad()

        features = features.to('cuda')
        labels = labels.to('cuda')
        
        y_pred = model(features)

        loss = criterion(y_pred, labels.reshape(-1, 1))
        loss.backward()

        optimizer.step()

        total_loss += loss.item()
    
    print(f"loss at epoch {i + 1}: ", total_loss / len(train_dataloader))

loss at epoch 1:  4.810903128816022
loss at epoch 2:  1.8310943010780547
loss at epoch 3:  1.801656660106447
loss at epoch 4:  1.782842053141859
loss at epoch 5:  1.7700971906383833
loss at epoch 6:  1.7630521782570414
loss at epoch 7:  1.7569076182113754
loss at epoch 8:  1.7531058539946873
loss at epoch 9:  1.750086782872677
loss at epoch 10:  1.7474791511893273
loss at epoch 11:  1.7456667166617181
loss at epoch 12:  1.7432123959064483
loss at epoch 13:  1.7427743615375624
loss at epoch 14:  1.7407429062657886
loss at epoch 15:  1.7400516423086325
loss at epoch 16:  1.739471936557028
loss at epoch 17:  1.738556132879522
loss at epoch 18:  1.737657873829206
loss at epoch 19:  1.7368497487571504
loss at epoch 20:  1.7353372154964342
loss at epoch 21:  1.7340539660718706
loss at epoch 22:  1.7350750881764623
loss at epoch 23:  1.7336472532815403
loss at epoch 24:  1.731944430536694
loss at epoch 25:  1.7323739000492626


In [23]:
model.eval()

CustomModel(
  (model): Sequential(
    (0): Linear(in_features=34, out_features=10, bias=True)
    (1): GELU(approximate='none')
    (2): Linear(in_features=10, out_features=5, bias=True)
    (3): GELU(approximate='none')
    (4): Linear(in_features=5, out_features=1, bias=True)
  )
)

In [24]:
from sklearn.metrics import r2_score

all_preds = []
all_labels = []

with torch.no_grad():
    for features, labels in test_dataloader:
        features = features.to('cuda')
        preds = model(features)
        
        all_preds.append(preds.cpu())
        all_labels.append(labels)

# Concatenate all batches
y_pred = torch.cat(all_preds).numpy().flatten()
y_true = torch.cat(all_labels).numpy().flatten()

# Calculate actual R2 Score
score = r2_score(y_true, y_pred)
print(f"R2 Score: {score:.4f}")

R2 Score: 0.5045


$R^2 = 1 - \frac{\sum (y_i - \hat{y}_i)^2}{\sum (y_i - \bar{y})^2}$